In [1]:
import scanpy as sc
import tempfile
import requests
import os 


temp_combined_path = os.path.join(tempfile.gettempdir(), "adata_combined.h5ad")
adata= sc.read_h5ad(temp_combined_path)

temp_cereb_path = os.path.join(tempfile.gettempdir(), "cereb_combined.h5ad")
cereb_data = sc.read_h5ad(temp_cereb_path)

In [2]:
shared_genes = list(set(adata.var_names) & set(cereb_data.var_names))
adata_sub = adata[:, shared_genes]
cereb_sub = cereb_data[:, shared_genes]

cereb_sub

View of AnnData object with n_obs × n_vars = 82862 × 120
    obs: 'orig_cluster', 'orig_sub_cluster', 'broad_lineage', 'author_cell_type', 'dev_state', 'subtype', 'precisest_label', 'tissue_id', 'batch', 'size_factor', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'is_primary_data', 'author_stage', 'tissue_fragment', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_counts', 'n_genes'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_cells'
    uns: 'batch_condition', 'cell_type_colors', 'citation', 'default_embedding', 'disease_colors', 'neighbors', 'organism', 'organism_ontology_term_id', 'pca', 'schema_reference', 'schema_ve

In [3]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

X_ad = adata_sub.X
y_ad = LabelEncoder().fit_transform(adata_sub.obs["disease"])  # e.g., AD vs Control

X_cereb = cereb_sub.X
if "development_stage" in cereb_sub.obs.columns:
    y_cereb = LabelEncoder().fit_transform(cereb_sub.obs["development_stage"])
else:
    y_cereb = np.zeros(X_cereb.shape[0])

print(X_ad)

#print(y_ad)

[[ 1.87939384 -0.28566793  1.53113334 ... -0.94200827 -0.80771344
   1.68508067]
 [ 2.19018843 -0.28566793 -0.99458796 ... -0.94200827 -0.80771344
  -0.73657895]
 [-0.60050543 -0.28566793  1.6975118  ...  1.77032425 -0.80771344
  -0.73657895]
 ...
 [-0.60050543 -0.28566793  0.68479065 ...  0.46019066  0.57967642
  -0.73657895]
 [ 0.97217396 -0.28566793  0.91565929 ...  0.43880695  0.99766623
   0.79916656]
 [-0.60050543 -0.28566793 -0.99458796 ... -0.94200827  0.71846657
   0.79994722]]


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectFromModel

# SVM for AD
svm_ad = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_ad.fit(X_ad, y_ad)
selected_ad = np.abs(svm_ad.coef_).sum(axis=0)

# SVM for Cerebellum
svm_cereb = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_cereb.fit(X_cereb, y_cereb)
selected_cereb = np.abs(svm_cereb.coef_).sum(axis=0)

# Top genes
top_n = 50
genes_array = np.array(shared_genes)
top_ad_genes = genes_array[np.argsort(selected_ad)[-top_n:]]
top_cereb_genes = genes_array[np.argsort(selected_cereb)[-top_n:]]
overlap = set(top_ad_genes) & set(top_cereb_genes)
print(f"Overlap ({len(overlap)}): {overlap}")


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Combine inputs
X_combined = np.vstack([X_ad, X_cereb])
y_ad_combined = np.concatenate([y_ad, [-1]*X_cereb.shape[0]])
y_cereb_combined = np.concatenate([[-1]*X_ad.shape[0], y_cereb])
# Split the data
X_ad_train, X_ad_test, y_ad_train, y_ad_test = train_test_split(X_ad, y_ad, test_size=0.2, random_state=42)
X_cereb_train, X_cereb_test, y_cereb_train, y_cereb_test = train_test_split(X_cereb, y_cereb, test_size=0.2, random_state=42)
# Train separate MLPs (simplified multitask)
mlp_ad = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)

mlp_ad.fit(X_ad_train, y_ad_train)
print("AD classification report:")
print(classification_report(y_ad_test, mlp_ad.predict(X_ad_test)))

mlp_cereb = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
mlp_cereb.fit(X_cereb_train, y_cereb_train)
print("Cerebellum classification report:")
print(classification_report(y_cereb_test, mlp_cereb.predict(X_cereb_test)))


AD classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6020
           1       1.00      1.00      1.00      8939
           2       1.00      1.00      1.00      9383

    accuracy                           1.00     24342
   macro avg       1.00      1.00      1.00     24342
weighted avg       1.00      1.00      1.00     24342

Cerebellum classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13445
           1       1.00      1.00      1.00     13197
           2       1.00      1.00      1.00      7849
           3       1.00      1.00      1.00     14200
           4       1.00      1.00      1.00     15580
           5       1.00      1.00      1.00      4594
           6       1.00      1.00      1.00      8033
           7       1.00      1.00      1.00      9359
           8       1.00      1.00      1.00      3181
           9      

In [ ]:
from sklearn.inspection import permutation_importance

# Convert to dense if needed
X_ad_test_array = X_ad_test.toarray() if hasattr(X_ad_test, "toarray") else X_ad_test
X_cereb_test_array = X_cereb_test.toarray() if hasattr(X_cereb_test, "toarray") else X_cereb_test

# Permutation importance on test sets only
perm_ad = permutation_importance(mlp_ad, X_ad_test_array, y_ad_test, n_repeats=10, random_state=42)
top_nn_ad = genes_array[np.argsort(perm_ad.importances_mean)[-top_n:]]

perm_cereb = permutation_importance(mlp_cereb, X_cereb_test_array, y_cereb_test, n_repeats=10, random_state=42)
top_nn_cereb = genes_array[np.argsort(perm_cereb.importances_mean)[-top_n:]]

# Overlapping genes in NN-based importance
overlap_nn = set(top_nn_ad) & set(top_nn_cereb)
print(f"Overlap in NN-important genes: {overlap_nn}")
